In [53]:
using LinearAlgebra
using Distributions
using LaTeXStrings
using Printf
using FileIO
import JLD2

In [54]:
using DataFrames

In [55]:
using Revise
using Newtrinos
using Newtrinos.osc

In [56]:
osc_cfg = Newtrinos.osc.OscillationConfig(
    flavour=Newtrinos.osc.NNM(),
    propagation=Newtrinos.osc.Basic(),
    states=Newtrinos.osc.All(),
    interaction=Newtrinos.osc.SI()
    )

osc = Newtrinos.osc.configure(osc_cfg)

Newtrinos.osc.Osc(OscillationConfig{NNM, SI, Basic, Newtrinos.osc.All}(NNM(ThreeFlavour(:NO)), SI(), Basic(), Newtrinos.osc.All()), (m₀ = 0.1, θ₁₂ = 0.5872523687443223, θ₁₃ = 0.1454258194533693, θ₂₃ = 0.8556288707523761, δCP = 1.0, Δm²₂₁ = 7.53e-5, Δm²₃₁ = 0.0024752999999999997, N = 100.0, r = 1.0), (m₀ = Uniform{Float64}(a=1.0e-7, b=0.03), θ₁₂ = Uniform{Float64}(a=0.4205343352839651, b=0.7853981633974483), θ₁₃ = Uniform{Float64}(a=0.1, b=0.2), θ₂₃ = Uniform{Float64}(a=0.5235987755982988, b=1.0471975511965976), δCP = Uniform{Float64}(a=0.0, b=6.283185307179586), Δm²₂₁ = Uniform{Float64}(a=6.5e-5, b=9.0e-5), Δm²₃₁ = Uniform{Float64}(a=0.002, b=0.003), N = Uniform{Float64}(a=1.0, b=40.0), r = Uniform{Float64}(a=0.0, b=1.0)), Newtrinos.osc.var"#get_Nnaturalness#126"(), Newtrinos.osc.var"#osc_prob#43"{Newtrinos.osc.var"#osc_prob#41#44"{OscillationConfig{NNM, SI, Basic, Newtrinos.osc.All}}}(Newtrinos.osc.var"#osc_prob#41#44"{OscillationConfig{NNM, SI, Basic, Newtrinos.osc.All}}(OscillationC

In [57]:

atm_flux = Newtrinos.atm_flux.configure()
earth_layers = Newtrinos.earth_layers.configure()

physics = (; osc, atm_flux, earth_layers);

In [58]:
experiments = (
 
    dayabay = Newtrinos.dayabay.configure(physics),
);

[ Info: Loading dayabay data


In [59]:
p = Newtrinos.get_params(experiments)

(N = 100.0, m₀ = 0.1, r = 1.0, Δm²₂₁ = 7.53e-5, Δm²₃₁ = 0.0024752999999999997, δCP = 1.0, θ₁₂ = 0.5872523687443223, θ₁₃ = 0.1454258194533693, θ₂₃ = 0.8556288707523761)

In [ ]:
using CairoMakie
E_range = range(0.002, 0.04, length=1000) 
L=1.663
L_vec = [L]


osc_prob = Newtrinos.osc.get_osc_prob(osc_cfg)

p= Newtrinos.get_params(experiments)
p_5 = merge(p, (N = ftype(100),))


probab_5 = osc_prob(collect(E_range), L_vec, p_5; anti=true)

fig = Figure(size=(800, 600))
ax = Axis(fig[1,1], 
    xlabel = "Energy (GeV)",
    ylabel = "P(ν̄ₑ → ν̄ₑ)", 
    title = "Oscillation Probabilities Dayabay - Dirac case - r=1"
)

# Plot the lines
#lines!(ax, E_range, probab_SM[:, 1, 1,1], label="SM", linewidth=2, color=:green)
lines!(ax, E_range, probab_5[:, 1, 1,1], label=" N=100", linewidth=2, color=:blue)


# Add legend and set limits
axislegend(ax, position=:rb)

display(fig)


In [ ]:
img = experiments.dayabay.plot(p)
display("image/png", img)
#save("/home/sofialon/Newtrinos.jl/profiled plot/dayabay/dayabay_data_NND_100_0.png", img)

In [ ]:

all_priors = Newtrinos.get_priors(experiments)


#vars_to_scan = (r=31,  m₀=31)  
vars_to_scan=(r=31,   N=31)

modified_priors = (
    N = all_priors.N,
    m₀ =all_priors.m₀,
    r = all_priors.r,
    
   
  

    Δm²₂₁ = p.Δm²₂₁,  
    Δm²₃₁ =all_priors.Δm²₃₁ , 
    δCP = p.δCP,    
    θ₁₂ = p.θ₁₂,    
    θ₁₃= all_priors.θ₁₃,       
    θ₂₃ = p.θ₂₃   
    

)


In [ ]:

likelihood = Newtrinos.generate_likelihood(experiments);


In [ ]:

result = Newtrinos.scan(likelihood, modified_priors, vars_to_scan, p)

In [ ]:
#JLD2.@save "scan_dayabay_rm₀_NND_no_scale.jld2" result

In [ ]:
using CairoMakie

In [ ]:
img = CairoMakie.plot(result)
display("image/png", img)
save("/home/sofialon/Newtrinos.jl/plots_22/scan_dayabay_Nr_NND_m0=0.01_damping.png", img)


In [ ]:
bf = Newtrinos.bestfit(result)

# Calculate confidence intervals
sigma_1_threshold = maximum(result.values.log_posterior[:, 15]) - 0.5
sigma_2_threshold = maximum(result.values.log_posterior[:, 15]) - 2.0

# Create the plot
fig = Figure(resolution = (800, 600))
ax = Axis(fig[1, 1],
    xlabel = "r",
    ylabel = "Log Posterior",
    title = "r vs Log posterior - Dayabay NNM",
    titlesize = 16,
    xlabelsize = 14,
    ylabelsize = 14
)

# Plot the main curve
lines!(ax, result.axes.r, result.values.log_posterior[:, 25],
    color = :blue,
    linewidth = 2,
    label = "Log Posterior"
)

# Add confidence level lines
hlines!(ax, [sigma_1_threshold], 
    color = :red, 
    linestyle = :dash, 
    linewidth = 2,
    label = "1σ"
)

hlines!(ax, [sigma_2_threshold], 
    color = :orange, 
    linestyle = :dash, 
    linewidth = 2,
    label = "2σ"
)

# Add legend
axislegend(ax, position = :rb)  # right bottom

display("image/png", fig)
#save("/home/sofialon/Newtrinos.jl/profiled plot/dayabay/dayabay_rLogpost_N=50_m0=0.1_NND.png", fig)


Oscillations


In [ ]:

# STANDARD MODEL
E_range = range(0.002, 0.04, length=1000) 
L=1.663
L_vec = [L]

osc_cfg_SM = Newtrinos.osc.OscillationConfig(
    flavour=Newtrinos.osc.ThreeFlavour(),
    propagation=Newtrinos.osc.Basic(),
    states=Newtrinos.osc.All(),
    interaction=Newtrinos.osc.SI()
    )

osc_SM = Newtrinos.osc.configure(osc_cfg_SM)
 

atm_flux = Newtrinos.atm_flux.configure()
earth_layers = Newtrinos.earth_layers.configure()

physics_SM = (; osc=osc_SM, atm_flux, earth_layers);


experiments_SM = (
 
    dayabay = Newtrinos.dayabay.configure(physics_SM),
);

p_SM = Newtrinos.get_params(experiments_SM)

osc_prob_SM = Newtrinos.osc.get_osc_prob(osc_cfg_SM)


probab_SM = osc_prob_SM(collect(E_range), L_vec, p; anti=true)

In [ ]:
# NNATURALNESS WITH DIFFERENT N 


osc_prob = Newtrinos.osc.get_osc_prob(osc_cfg)

p= Newtrinos.get_params(experiments)
p_5 = merge(p, (N = ftype(5), r= ftype(1),))
p_10 = merge(p, (N = ftype(10),r= ftype(1),))
p_20 = merge(p, (N = ftype(20),r= ftype(1),))
p_50 = merge(p, (N = ftype(50),r= ftype(1),))

probab_5 = osc_prob(collect(E_range), L_vec, p_5; anti=true)
probab_10 = osc_prob(collect(E_range), L_vec, p_10; anti=true)
probab_20 = osc_prob(collect(E_range), L_vec, p_20; anti=true)
probab_50 = osc_prob(collect(E_range), L_vec, p_50; anti=true)

p_0 = merge(p, (r = ftype(0),))
p_025 = merge(p,  (r = ftype(0.25),))
p_05 = merge(p,  (r = ftype(0.5),))
p_1 = merge(p,  (r = ftype(1),))


probab_0 = osc_prob(collect(E_range), L_vec, p_0; anti=true)
probab_025 = osc_prob(collect(E_range), L_vec, p_025; anti=true)
probab_05 = osc_prob(collect(E_range), L_vec, p_05; anti=true)
probab_1 = osc_prob(collect(E_range), L_vec, p_1; anti=true)


In [ ]:
using CairoMakie

In [ ]:

fig = Figure(size=(800, 600))
ax = Axis(fig[1,1], 
    xlabel = "Energy (GeV)",
    ylabel = "P(ν̄ₑ → ν̄ₑ)", 
    title = "Oscillation Probabilities Dayabay - Majorana case - r=1"
)

# Plot the lines
lines!(ax, E_range, probab_SM[:, 1, 1,1], label="SM", linewidth=2, color=:green)
lines!(ax, E_range, probab_5[:, 1, 1,1], label=" N=5", linewidth=2, color=:blue)
lines!(ax, E_range, probab_10[:, 1, 1,1], label="N=10", linewidth=2, color=:red)
lines!(ax, E_range, probab_20[:, 1,1,1], label=" N=20", linewidth=2, color=:purple)
lines!(ax, E_range, probab_50[:, 1,1,1], label=" N=50", linewidth=2, color=:orange)

# Add legend and set limits
axislegend(ax, position=:rb)
#save("/home/sofialon/Newtrinos.jl/plots_22/osc_dayabay_N_NNM_m0=0.1_r=1.png", fig)

fig

In [ ]:

fig = Figure(size=(800, 600))
ax = Axis(fig[1,1], 
    xlabel = "Energy (GeV)",
    ylabel = "Oscillation Probability", 
    title = "Oscillation Probabilities Dayabay - Majorana case - N=100 "
)

# Plot the lines
lines!(ax, E_range, probab_SM[:, 1, 1, 1], label="P(νₑ → νₑ) SM", linewidth=2, color=:green)
lines!(ax, E_range, probab_0[:, 1, 1, 1], label="P(νₑ → νₑ) r=0", linewidth=2, color=:blue)
lines!(ax, E_range, probab_025[:, 1, 1, 1], label="P(νₑ →  νₑ) r=0.25", linewidth=2, color=:red)
lines!(ax, E_range, probab_05[:, 1, 1, 1], label="P(νₑ →  νₑ) r=0.5", linewidth=2, color=:purple)
lines!(ax, E_range, probab_1[:, 1, 1, 1], label="P(νₑ →  νₑ) r=1", linewidth=2, color=:orange)

# Add legend and set limits
axislegend(ax, position=:rb)
#save("/home/sofialon/Newtrinos.jl/new_plots/osc_dayabay_r_NNM_m0=0.1_N=100.png", fig)

fig

In [ ]:
using CairoMakie
f1 = Newtrinos.osc.get_matrices(osc_cfg.flavour)
    

params_temp = merge(p, (N=50, r =1, m₀=0.3))
H, FinalU, h_temp, norm_sector, norm_gamma, gamma, gamma_sq = f1(params_temp)

params_temp_1= merge(p, (N=50, r =1, m₀=0.01))
H_1, FinalU_1, h_temp_1, norm_sector, norm_gamma, gamma, gamma_sq  = f1(params_temp_1)


fig = Figure(size=(1200,500))
ax1 = Axis(fig[1, 1])
hm = heatmap!(ax1, abs.(H), colormap=:viridis)
Colorbar(fig[1, 2], hm, label="Magnitude")
ax2 = Axis(fig[1,3])
hm = heatmap!(ax2, abs.(H_1), colormap=:viridis)
Colorbar(fig[1, 4], hm, label="Magnitude")

ax1.title = " Perturbation m0=0.3 - N=50"
ax2.title = " Perturbation m0.01 - N=50"

fig
    

#save("Mpert_50_NNM.png", fig)

Gamma values: 0.02, 0.020008364917370604, 0.0202731678169282
Gamma values: 0.0004, 0.00040033466666666677, 0.00041100133333333335
m1 = 0.3
m2 = 0.30012547376055904
m3 = 0.304097517253923
scale15.0
Ratio ||M_gamma_factor|| / ||M_sector_part|| = 1.605634891100713e-5
0.0
7.53e-5
0.0024752999999999997
Gamma values: 0.02, 0.02648018126826174, 0.10149482745440773
Gamma values: 0.0004, 0.0007012, 0.010301199999999996
m1 = 0.01
m2 = 0.01324009063413087
m3 = 0.050747413727203865
scale0.5
Ratio ||M_gamma_factor|| / ||M_sector_part|| = 8.03839033438909e-5
0.0
7.53e-5
0.0024752999999999997


Eigenvalues plotting

In [63]:


# Plot 1: All eigenvalues vs m1 (scatter plot showing full structure)
function plot_eigenvalues_vs_m1(m1_values, all_eigenvalues)
    fig = Figure(size=(1200, 700))
    ax = Axis(fig[1, 1],
        xlabel="m₁ (eV)",
        ylabel="Eigenvalue",
        title="Full Eigenvalue Spectrum vs m₁ - N=10",
        yscale=log10)
    
    colors = [:blue, :red, :green, :orange, :purple, :brown, :pink, :cyan, :magenta, :yellow]
    
    # Plot eigenvalues at each m1 with color coding
    for (i, m1) in enumerate(m1_values)
        eigs = all_eigenvalues[i]  # This is a 1D vector
        if !isempty(eigs)
            color = colors[mod(i-1, length(colors)) + 1]
            # Plot each eigenvalue for this m1
            scatter!(ax, fill(m1, length(eigs)), eigs, 
                markersize=8, color=color, alpha=0.7, 
                label="m₁ = $(round(m1, sigdigits=2))")
        end
    end
    
    axislegend(ax, position=:lt, nbanks=2)
    fig
end


# Plot 2: Eigenvalue ratios vs eigenvalue index (colored by m1)
function plot_eigenvalue_ratios(m1_values, all_ratios)
    fig = Figure(size=(1200, 700))
    ax = Axis(fig[1, 1],
        xlabel="Eigenvalue Index Ratio (i+1 / i)",
        ylabel="Ratio Value",
        title="Consecutive Eigenvalue Ratios - N=10",
        yscale=log10)
    
    colors = [:blue, :red, :green, :orange, :purple, :brown, :pink, :cyan, :magenta, :yellow]
    
    # Each m1 value gets its own color
    for (i, m1) in enumerate(m1_values)
        ratios = all_ratios[i]
        if !isempty(ratios)
            color = colors[mod(i-1, length(colors)) + 1]
            # Plot as line
            lines!(ax, 1:length(ratios), ratios, 
                   label="m₁ = $(round(m1, sigdigits=2))", 
                   linewidth=2.5, color=color)
            # Add markers separately
            scatter!(ax, 1:length(ratios), ratios,
                    markersize=7, color=color)
        end
    end
    
    # Add reference line at y=1 (no hierarchy)
    hlines!(ax, [1.0], color=:black, linestyle=:dash, linewidth=2, label="No structure", alpha=0.5)
    
    axislegend(ax, position=:rt, nbanks=2)
    fig
end


# Plot 3: Perturbation parameter vs m1
function plot_perturbation_parameter(m1_values, all_norm_ratios)
    fig = Figure(size=(1000, 700))
    ax = Axis(fig[1, 1],
        xlabel="m₁ (eV)",
        ylabel="||M_gamma_factor|| / ||M_sector_part||",
        title="Perturbation Validity - N=10",
        yscale=log10
    )
    
    # Filter out NaN values
    valid_idx = .!isnan.(all_norm_ratios)
    valid_m1 = m1_values[valid_idx]
    valid_ratios = all_norm_ratios[valid_idx]
    
    if !isempty(valid_ratios)
        lines!(ax, valid_m1, valid_ratios, linewidth=3, color=:blue, label="Norm ratio")
        scatter!(ax, valid_m1, valid_ratios, markersize=10, color=:blue)
        
        # Add critical threshold lines
        hlines!(ax, [0.01], color=:green, linestyle=:dash, linewidth=2.5, 
                label="Safe perturbation (< 0.1)", alpha=0.7)
        hlines!(ax, [0.03], color=:orange, linestyle=:dash, linewidth=2.5, 
                label="Marginal (0.1-0.3)", alpha=0.7)
        hlines!(ax, [1.0], color=:red, linestyle=:dash, linewidth=2.5, 
                label="Breakdown (> 1)", alpha=0.7)
    end
    
    axislegend(ax, position=:rt)
    fig
end


# Main analysis function
function analyze_m1_limit(base_params, m1_values)
    """
    Analyze eigenvalue structure vs m1 to find the perturbation limit
    
    Args:
        base_params: NamedTuple with base parameters
        m1_values: Vector of m1 values to scan
        get_Nnaturalness_func: Your function that computes eigenvalues
        N_int: Number of sectors
        r: Sector parameter
    """
    
    all_eigenvalues = []
    all_ratios = []
    all_norm_ratios = []
    
    for m1_test in m1_values
        println("Computing for m1 = $m1_test")
        
        # Update parameters with new m1
        params_test = merge(base_params, (m₀=m1_test,))
        
        try
            # Call your function - it should return eigenvalues
            f1 = Newtrinos.osc.get_matrices(osc_cfg.flavour)
            EIG, FinalU, h, norm_sector, norm_gamma = f1(params_test)
            
            # Extract ALL eigenvalues from the full matrix
            # Assuming EIG is a Diagonal matrix or eigenvalues vector
            if isa(EIG, Diagonal)
                eigs_sorted = sort(real.(diag(EIG)))
            else
                eigs_sorted = sort(real.(EIG))
            end
            
            push!(all_eigenvalues, eigs_sorted)
            
            # Compute ratios of consecutive eigenvalues
            if length(eigs_sorted) > 1
                ratios = eigs_sorted[2:end] ./ eigs_sorted[1:end-1]
                push!(all_ratios, ratios)
            else
                push!(all_ratios, [])
            end
            
            # Compute norm ratio (perturbation parameter)
            norm_ratio = norm_gamma / norm_sector
            push!(all_norm_ratios, norm_ratio)
            
            println("  -> $(length(eigs_sorted)) eigenvalues, norm ratio = $(round(norm_ratio, sigdigits=3))")
            
        catch e
            println("  Error: $e")
            push!(all_eigenvalues, [])
            push!(all_ratios, [])
            push!(all_norm_ratios, NaN)
        end
    end
    
    return all_eigenvalues, all_ratios, all_norm_ratios
end



function main(N_int)
    """
    Main function to run the analysis
    
    Usage: main(get_Nnaturalness)
    Pass your get_Nnaturalness function as argument
    """
    N=N_int
    
    # MODIFY THESE WITH YOUR ACTUAL VALUES
    base_params =merge(p, (N=10, r=1))
    
    # Logarithmic scan of m1 values
    m1_values = LinRange(0.0001, 0.0005,10)
    
   
    println("EIGENVALUE STRUCTURE ANALYSIS")
    
    println("Scanning m1 values: $(round.(m1_values, sigdigits=2))\n")
    
    all_eigenvalues, all_ratios, all_norm_ratios = analyze_m1_limit(
        base_params, m1_values
    )
    
    println("GENERATING PLOTS")
    
    
    # Generate Plot 1: Eigenvalues vs m1
    println("Generating Plot 1: Eigenvalues vs m1...")
    fig1 = plot_eigenvalues_vs_m1(m1_values, all_eigenvalues)
    save("eigenvalues_vs_m1_ex_$N.png", fig1)
    println("✓ Saved: eigenvalues_vs_m1_ex.png\n")
    
    # Generate Plot 2: Eigenvalue ratios
    println("Generating Plot 2: Eigenvalue ratios...")
    fig2 = plot_eigenvalue_ratios(m1_values, all_ratios)
    save("eigenvalue_ratios_vs_m1_ex_$N.png", fig2)
    println("✓ Saved: eigenvalue_ratios_vs_m1.png\n")
    
    # Generate Plot 3: Perturbation parameter
    if !isempty(all_norm_ratios) && !all(isnan.(all_norm_ratios))
        println("Generating Plot 3: Perturbation parameter...")
        fig3 = plot_perturbation_parameter(m1_values, all_norm_ratios)
        save("perturbation_parameter_vs_m1_ex_$N.png", fig3)
        println("✓ Saved: perturbation_parameter_vs_m1.png\n")
    else
        println("⚠ Skipping perturbation plot - no valid norm ratios\n")
    end

    return fig1, fig2, all_eigenvalues, all_ratios, all_norm_ratios
end


# To use this script:
# 1. Make sure your get_Nnaturalness function returns:
#    EIG, FinalUmatrix, h, norm_sector, norm_gamma
# 
fig1, fig2, eigs, ratios, norms = main(10)
#
# Color scheme:
# - Blue: smallest m1 values
# - Red, Green, Orange, etc: progressively larger m1 values
# - Each color appears consistently across all three plots

EIGENVALUE STRUCTURE ANALYSIS
Scanning m1 values: [0.0001, 0.00014, 0.00019, 0.00023, 0.00028, 0.00032, 0.00037, 0.00041, 0.00046, 0.0005]

Computing for m1 = 0.0001
Gamma values: 0.1, 8.678133439859057, 49.75248737500468
Gamma values: 0.010000000000000002, 75.30999999999999, 2475.31
m1 = 0.0001
m2 = 0.008678133439859056
m3 = 0.04975248737500468
scale0.001
Ratio ||M_gamma_factor|| / ||M_sector_part|| = 0.9452972601250889
0.0
7.53e-5
0.0024752999999999997
  -> 30 eigenvalues, norm ratio = 0.945
Computing for m1 = 0.00014444444444444444
Gamma values: 0.1, 6.008371871345739, 34.44410530892832
Gamma values: 0.010000000000000002, 36.100532544378694, 1186.3963905325443
m1 = 0.00014444444444444444
m2 = 0.008678759369721623
m3 = 0.049752596557340906
scale0.0014444444444444444
Ratio ||M_gamma_factor|| / ||M_sector_part|| = 0.6544380008696382
0.0
7.53e-5
0.0024752999999999997
  -> 30 eigenvalues, norm ratio = 0.654
Computing for m1 = 0.00018888888888888888
Gamma values: 0.1, 4.595089149369949, 2

(Scene (1200px, 700px):
  0 Plots
  2 Child Scenes:
    ├ Scene (1200px, 700px)
    └ Scene (1200px, 700px), Scene (1200px, 700px):
  0 Plots
  2 Child Scenes:
    ├ Scene (1200px, 700px)
    └ Scene (1200px, 700px), Any[[1.1975947436801159e-8, 1.0522039902010082e-7, 2.887706420228307e-7, 5.619121473990917e-7, 9.248281357342487e-7, 1.3784709048664301e-6, 1.9249378555207226e-6, 2.5688441013155283e-6, 3.323474706454041e-6, 8.904680741188617e-5  …  0.06899113473482683, 0.13489005761002862, 0.2228496087223175, 0.3332578813027426, 0.4668440731558, 0.6251119117613128, 0.8121938272330633, 1.0026622115651584, 1.247790042949706, 2.923669201512404], [2.4986853294071635e-8, 2.1953391894321121e-7, 6.024967716277644e-7, 1.1723846038327248e-6, 1.929579690605993e-6, 2.8760689249679014e-6, 4.0162283652218075e-6, 5.359687075583013e-6, 6.934163276430289e-6, 8.939676468850776e-5  …  0.06962194850648662, 0.1359928475465351, 0.2245007291579685, 0.3355001106768907, 0.46966662230501954, 0.6283980003905868, 0

Gamma plotting

In [65]:


function plot_gamma_values(m1_values, all_gamma, all_gamma_squared)
    """
    Plot gamma and gamma^2 values vs m1
    
    Args:
        m1_values: Vector of m1 values
        all_gamma: Vector of gamma vectors (3 values each)
        all_gamma_squared: Vector of gamma^2 vectors (3 values each)
    """
    
    fig = Figure(size=(1400, 600))
    
    # Plot 1: gamma values
    ax1 = Axis(fig[1, 1],
        xlabel="m₁ (eV)",
        ylabel="γ value",
        title="γ vs m₁ - N=10",
        yscale=log10)
    
    colors = [:blue, :red, :green]
    labels = ["γ₁ = m₁²/m₁²", "γ₂ = m₂²/m₁²", "γ₃ = m₃²/m₁²"]
    
    # Plot each gamma component
    for component in 1:3
        gamma_component = [all_gamma[i][component] for i in 1:length(m1_values)]
        lines!(ax1, m1_values, gamma_component, 
               linewidth=2.5, color=colors[component], label=labels[component])
        scatter!(ax1, m1_values, gamma_component, 
                markersize=8, color=colors[component])
    end
    
    axislegend(ax1, position=:rt)
    
    # Plot 2: gamma^2 values
    ax2 = Axis(fig[1, 2],
        xlabel="m₁ (eV)",
        ylabel="γ² value",
        title="γ² vs m₁  - N=50",
        yscale=log10)
    
    labels_sq = ["γ₁² = m₁⁴/m₁⁴", "γ₂² = m₂⁴/m₁⁴", "γ₃² = m₃⁴/m₁⁴"]
    
    # Plot each gamma^2 component
    for component in 1:3
        gamma_sq_component = [all_gamma_squared[i][component] for i in 1:length(m1_values)]
        lines!(ax2, m1_values, gamma_sq_component, 
               linewidth=2.5, color=colors[component], label=labels_sq[component])
        scatter!(ax2, m1_values, gamma_sq_component, 
                markersize=8, color=colors[component])
    end
    
    axislegend(ax2, position=:rt)
    
    fig
end


function plot_gamma_ratio(m1_values, all_gamma, all_gamma_squared)
    """
    Plot ratio γ²/γ for each component (should equal γ)
    """
    
    fig = Figure(size=(1200, 600))
    
    # Plot 1: Ratio γ²/γ
    ax1 = Axis(fig[1, 1],
        xlabel="m₁ (eV)",
        ylabel="γ²/γ",
        title="Ratio γ²/γ vs m₁  - N=50",
        yscale=log10)
    
    colors = [:blue, :red, :green]
    labels = ["γ₁²/γ₁", "γ₂²/γ₂", "γ₃²/γ₃"]
    
    # Compute ratios γ²/γ for each component
    for component in 1:3
        ratio_component = [all_gamma_squared[i][component] / all_gamma[i][component] 
                          for i in 1:length(m1_values)]
        lines!(ax1, m1_values, ratio_component, 
               linewidth=2.5, color=colors[component], label=labels[component])
        scatter!(ax1, m1_values, ratio_component, 
                markersize=8, color=colors[component])
    end
    
    axislegend(ax1, position=:rt)
    
    # Plot 2: Direct comparison γ vs γ²/γ (verification they're the same)
   #= ax2 = Axis(fig[1, 2],
        xlabel="m₁ (GeV)",
        ylabel="Value",
        title="Verification: γ (line) vs γ²/γ (scatter)",
        xscale=log10,
        yscale=log10)
    
    for component in 1:3
        gamma_component = [all_gamma[i][component] for i in 1:length(m1_values)]
        ratio_component = [all_gamma_squared[i][component] / all_gamma[i][component] 
                          for i in 1:length(m1_values)]
        
        # Plot γ as line
        lines!(ax2, m1_values, gamma_component, 
               linewidth=2.5, color=colors[component], linestyle=:solid, alpha=0.7)
        # Plot γ²/γ as scatter to verify they match
        scatter!(ax2, m1_values, ratio_component, 
                markersize=6, color=colors[component], marker=:circle)
    end
    =#
    fig
end


# Example usage:
function main_gamma(m1_values, base_params)
    """
    Main function to analyze gamma values
    
    Usage: main_gamma(m1_values, get_Nnaturalness, base_params)
    """
    
    all_gamma = []
    all_gamma_squared = []
    
    println("Computing gamma values for different m1...")
    
    for m1_test in m1_values
        params_test = merge(base_params, (m₀=m1_test,))
        
        try
            # Your function should return gamma values
            # Modify this based on what your get_Nnaturalness returns
            f1 = Newtrinos.osc.get_matrices(osc_cfg.flavour)
            EIG, FinalU, h, norm_sector, norm_gamma, gamma, gamma_sq = f1(params_test)
          
            
            push!(all_gamma, gamma)
            push!(all_gamma_squared, gamma_sq)
            
            println("m1 = $m1_test: γ = $gamma")
            
        catch e
            println("Error for m1 = $m1_test: $e")
        end
    end
    
    # Generate plots
    fig1 = plot_gamma_values(m1_values, all_gamma, all_gamma_squared)
    save("gamma_values_vs_m1_N50_NNM.png", fig1)
    println("\n✓ Saved: gamma_values_vs_m1_N50_NNM.png")

    fig2 = plot_gamma_ratio(m1_values, all_gamma, all_gamma_squared)
    save("gamma_ratios_vs_m1_N50_NNM.png", fig2)
    println("✓ Saved: gamma_ratios_vs_m1_N50_NNM.png")
    
    return fig1, fig2, all_gamma, all_gamma_squared
end
   
# MODIFY THESE WITH YOUR ACTUAL VALUES
base_params =merge(p, (N=50, r=1))

# Logarithmic scan of m1 values
m1_values = LinRange(0.01, 0.05,10)

   
fig1, fig2, all_gamma, all_gamma_squared=main_gamma(m1_values, p)

Computing gamma values for different m1...
Gamma values: 0.01, 0.01324009063413087, 0.050747413727203865
Gamma values: 0.0001, 0.0001753, 0.002575299999999999
m1 = 0.01
m2 = 0.01324009063413087
m3 = 0.050747413727203865
scale1.0
Ratio ||M_gamma_factor|| / ||M_sector_part|| = 1.0098735331713568e-5
0.0
7.53e-5
0.0024752999999999997
m1 = 0.01: γ = [0.01, 0.01324009063413087, 0.050747413727203865]
Gamma values: 0.01, 0.011665784694754943, 0.03586622910946374
Gamma values: 0.0001, 0.0001360905325443787, 0.0012863863905325443
m1 = 0.014444444444444444
m2 = 0.016850577892423807
m3 = 0.05180677538033651
scale1.4444444444444444
Ratio ||M_gamma_factor|| / ||M_sector_part|| = 7.137379592783284e-6
0.0
7.53e-5
0.0024752999999999997
m1 = 0.014444444444444444: γ = [0.01, 0.011665784694754943, 0.03586622910946374]
Gamma values: 0.01, 0.011004764617685263, 0.028173909990490298
Gamma values: 0.0001, 0.00012110484429065746, 0.000793769204152249
m1 = 0.01888888888888889
m2 = 0.020786777611183273
m3 = 0.05

(Scene (1400px, 600px):
  0 Plots
  4 Child Scenes:
    ├ Scene (1400px, 600px)
    ├ Scene (1400px, 600px)
    ├ Scene (1400px, 600px)
    └ Scene (1400px, 600px), Scene (1200px, 600px):
  0 Plots
  2 Child Scenes:
    ├ Scene (1200px, 600px)
    └ Scene (1200px, 600px), Any[[0.01, 0.01324009063413087, 0.050747413727203865], [0.01, 0.011665784694754943, 0.03586622910946374], [0.01, 0.011004764617685263, 0.028173909990490298], [0.01, 0.010669142994865988, 0.02355094347951925], [0.01, 0.010476587230582295, 0.02051338294869961], [0.01, 0.010356275275132285, 0.018395810022859723], [0.01, 0.01027622627457575, 0.01685565849026658], [0.01, 0.01022033736411753, 0.015698941262287616], [0.01, 0.010179802354008966, 0.014807897389707449], [0.01, 0.010149482745440775, 0.014107161301977091]], Any[[0.0001, 0.0001753, 0.002575299999999999], [0.0001, 0.0001360905325443787, 0.0012863863905325443], [0.0001, 0.00012110484429065746, 0.000793769204152249], [0.0001, 0.00011383061224489798, 0.000554646938775